# Break and Retest on SPY
## Strategy Brief
The Break and Retest strategy focuses on identifying when the SPY price breaks through a significant support or resistance level and then retests that level. The prediction is that if the price successfully retests the broken level, it will continue in the direction of the break. The trade logic involves entering a position when the price retests the level and shows signs of continuation. Historical testing of this strategy can provide insights into its effectiveness compared to a buy-and-hold approach.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
In this phase, we define the parameters required for the Break and Retest strategy. These include the lookback period for identifying significant levels and the threshold for considering a retest successful.

In [ ]:
LOOKBACK_PERIOD = 20  # Lookback period for identifying levels
RETEST_THRESHOLD = 0.01  # 1% threshold for retest
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'

### PHASE 2 - Data Exploration
We will download historical SPY data from Yahoo Finance and compute key indicators to identify significant levels. The price data will be plotted with these levels to visualize potential break and retest points.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)
data['20_day_high'] = data['High'].rolling(window=LOOKBACK_PERIOD).max()
data['20_day_low'] = data['Low'].rolling(window=LOOKBACK_PERIOD).min()

# Plot price and levels
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.plot(data['20_day_high'], label='20 Day High', linestyle='--')
plt.plot(data['20_day_low'], label='20 Day Low', linestyle='--')
plt.title('SPY Price with 20 Day High/Low')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
We define the signal for entering and exiting trades based on the break and retest logic. This involves creating a signal series that identifies when a break and retest occurs.

In [ ]:
data['Signal'] = 0

# Define break and retest logic
data['Break_High'] = (data['Close'] > data['20_day_high'].shift(1))
data['Retest_High'] = (data['Low'] <= data['20_day_high'].shift(1) * (1 + RETEST_THRESHOLD))
data.loc[data['Break_High'] & data['Retest_High'], 'Signal'] = 1

data['Break_Low'] = (data['Close'] < data['20_day_low'].shift(1))
data['Retest_Low'] = (data['High'] >= data['20_day_low'].shift(1) * (1 - RETEST_THRESHOLD))
data.loc[data['Break_Low'] & data['Retest_Low'], 'Signal'] = -1

# Position logic
data['Position'] = data['Signal'].replace(0, np.nan).ffill().fillna(0)

### PHASE 4 - Coding & Backtesting
We will backtest the strategy by shifting positions, calculating daily returns, and plotting the equity curve to visualize performance over time.

In [ ]:
# Calculate daily returns
data['Market_Return'] = data['Close'].pct_change()
data['Strategy_Return'] = data['Position'].shift(1) * data['Market_Return']
data['Equity_Curve'] = (1 + data['Strategy_Return']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity_Curve'], label='Strategy Equity Curve')
plt.title('Equity Curve of Break and Retest Strategy')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
We will evaluate the strategy's performance using key metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and Maximum Drawdown, and compare it to a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(equity_curve, market_return):
    cagr = (equity_curve.iloc[-1] / equity_curve.iloc[0]) ** (252 / len(equity_curve)) - 1
    annual_volatility = market_return.std() * np.sqrt(252)
    sharpe_ratio = (cagr - 0.02) / annual_volatility
    downside_volatility = market_return[market_return < 0].std() * np.sqrt(252)
    sortino_ratio = (cagr - 0.02) / downside_volatility
    max_drawdown = (equity_curve / equity_curve.cummax() - 1).min()
    calmar_ratio = cagr / abs(max_drawdown)
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

strategy_metrics = calculate_performance_metrics(data['Equity_Curve'], data['Strategy_Return'])
buy_and_hold_metrics = calculate_performance_metrics((1 + data['Market_Return']).cumprod(), data['Market_Return'])

# Display comparison table
comparison_table = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': strategy_metrics,
    'Buy and Hold': buy_and_hold_metrics
})
print(comparison_table)

### PHASE 6 - Deploy & Monitor
We create a function to download the last 60 days of SPY data, compute today's signal, and print the position to be taken.

In [ ]:
def get_latest_signal():
    recent_data = yf.download('SPY', period='60d')
    recent_data['20_day_high'] = recent_data['High'].rolling(window=LOOKBACK_PERIOD).max()
    recent_data['20_day_low'] = recent_data['Low'].rolling(window=LOOKBACK_PERIOD).min()
    
    recent_data['Signal'] = 0
    recent_data['Break_High'] = (recent_data['Close'] > recent_data['20_day_high'].shift(1))
    recent_data['Retest_High'] = (recent_data['Low'] <= recent_data['20_day_high'].shift(1) * (1 + RETEST_THRESHOLD))
    recent_data.loc[recent_data['Break_High'] & recent_data['Retest_High'], 'Signal'] = 1
    
    recent_data['Break_Low'] = (recent_data['Close'] < recent_data['20_day_low'].shift(1))
    recent_data['Retest_Low'] = (recent_data['High'] >= recent_data['20_day_low'].shift(1) * (1 - RETEST_THRESHOLD))
    recent_data.loc[recent_data['Break_Low'] & recent_data['Retest_Low'], 'Signal'] = -1
    
    position = recent_data['Signal'].iloc[-1]
    print(f"Today's position: {'Long' if position == 1 else 'Short' if position == -1 else 'Neutral'}")

get_latest_signal()